# 初期セットアップ（ステップ４）
このノートブックでは、サンプルコードの実行環境を初めて使用する際の設定のステップ４を行います。  
ステップ４では、ドメイン内にサンプルコードで共通して使われるユーザやコントラクトを作成します。  
作成したユーザに対応するウォレットファイルは実行環境のwalletsフォルダに格納されます。

## 必要なライブラリの読み込み

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { domain } = await import('../lib/load-config.mjs');
var { adminWallet, rpc, createObjectF, deploySmartContract } = await import('../lib/notebook-util.mjs');
var { loadWallet } = await import('../lib/load-wallet.mjs');

## 汎用コントラクトのデプロイ

In [2]:
var cid = await deploySmartContract({text: 'string'}, function _eval(text){
    setApiDelegation(true);
    return eval(text);
});

ドメイン内からアクセスできるようにします。

In [3]:
var resp = await rpc.call(adminWallet, 'c1update', { id: cid, prop: 'add accessible_to', value: [ domain ] });
console.log(resp);

{
  txno: 703001,
  txid: 'xxZZvtS89uRabK2Y6MyDnnBqDskHzVQdyhkUsVZYktr5MB',
  status: 'ok',
  value: null
}


## 動作確認用のユーザ作成

ウォレットを10個作成します。(作成済みであれば、再作成しません）

In [4]:
var fs = await import('node:fs');
var path = await import('node:path');
var { default: package_root } = await import('../lib/get-package-root.mjs');
for (var i=0; i<10; i++){
    var wallet_filename = path.join(package_root, 'wallets', `wallet-user${i}.json`);
    if(fs.existsSync(wallet_filename)) continue;
    var wfstr = await api.createWalletFile('user'+i, '_paswrd_', 'es');
    fs.writeFileSync(wallet_filename, wfstr, 'utf8');
}

作成したウォレットをユーザ登録します。（登録済みであれば、再登録しません）

In [5]:
for (var i=0; i<10; i++){
    var wfstr = await loadWallet(`wallet-user${i}.json`);
    var wallet = await api.unlockWalletFile(await api.parseWalletFile(wfstr), '_paswrd_');
    var id = (await rpc.call(wallet, 'c1query', { type: 'search', key: 'me' })).value[0].id;
    if (id === 'anonymous') {
        var id = await createObjectF('user', '_test_user'+i);
        await rpc.call(adminWallet, 'c1update', { id, prop: 'wallets', value: [wallet.address] });
    }
    console.log(id, wallet.address);
}
console.log('DONE');

u79273820 eQiA3bsfZunQ8KsRpkTgMGndwu46rq
u28344802 epGyEm4BXFxFvLmXkDdDvcqspESPFu
u56223755 eaZMspgRpi68EdReTZEiUkdJuKsdct
u08813170 e5fVm5R4eKYXa7U57pshS4UQwuvGsp
u06655488 eEYRycGm4znXxnf7bTngX72JmYk57r
u80054183 eWW6LCfRSpoVuudzZHa4ZwCjkkizov
u74989463 eYKwuqvDkdBAktANt6oebrdYmStexw
u49852900 eZbEh77Aq8RpmpZ6bgcTEYVGiYYNsv
u37771674 eX5jgq78UcBfUZg9SmnXFim6zLnf9r
u59975398 eAvw2Hf4TSreW2Sfj3PFUmXADUSZJg
DONE
